In [14]:
import os
print(os.getcwd())

/Users/hedanni/Desktop/DEPA_downstream


In [15]:
# Cell 1: 加载autoreload机制 + 所有需要的库
%load_ext autoreload
%autoreload 2

import yaml
from pathlib import Path
from depa_downstream import (
    load_cnmf_outputs,
    compute_program_direction, plot_program_direction_summary,
    compute_leading_genes,
    compute_foreground_gene_directions, plot_leading_genes_bar,
    compute_usage_binned_wpsi,subset_data_to_celltype,plot_usage_binned_trend,
    compute_contrast_association, plot_association_heatmap, plot_program_umap_grid,
    compute_gene_celltype_wpsi, plot_gene_celltype_heatmap,
    compute_program_gene_sets, select_program_markers,
    run_enrichment, plot_enrichment_bubble,
    extract_enrichment_driver_genes, plot_driver_gene_heatmap,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
# Cell 2: 读CONFIG
PROJECT_ROOT = Path.cwd()
CONFIG = yaml.safe_load(open(PROJECT_ROOT / "configs" / "covid19.yaml"))

In [17]:
import os

PATH_KEYS = [
    "cnmf_dir", "h5ad_path", "feature_anno_path",
    "wpsi_matrix_path", "psi_matrix_path",
    "depa_feature_source_path", "gene_symbol_map_path",
    "output_root",
]

def resolve_paths(config: dict) -> dict:
    root = os.path.expanduser(config["root_dir"])
    for key in PATH_KEYS:
        val = config.get(key)
        if not val:                      # 可选项没填就跳过
            continue
        if not os.path.isabs(val):       # 已经是绝对路径的保持不变
            resolved = os.path.join(root, val)
            if val.endswith("/"):        # 保留末尾斜杠
                resolved = resolved.rstrip("/") + "/"
            config[key] = resolved
    return config

CONFIG = resolve_paths(CONFIG)

In [18]:
# Cell 3: 读数据（这一步比较慢，几十秒到1分钟左右，因为要读wpsi/psi这种几个GB的大矩阵）
data = load_cnmf_outputs(CONFIG)

检查 cNMF 输入文件...
  [OK] spectra_score: /Users/hedanni/Desktop/DEPA_downstream/data/covid_19_data/cnmf_output_identity_state/COVID_DEAPA_programs/COVID_DEAPA_programs.gene_spectra_score.k_6.dt_0_01.txt (0.2 MB)
  [OK] spectra_tpm: /Users/hedanni/Desktop/DEPA_downstream/data/covid_19_data/cnmf_output_identity_state/COVID_DEAPA_programs/COVID_DEAPA_programs.gene_spectra_tpm.k_6.dt_0_01.txt (0.1 MB)
  [OK] usage: /Users/hedanni/Desktop/DEPA_downstream/data/covid_19_data/cnmf_output_identity_state/COVID_DEAPA_programs/COVID_DEAPA_programs.usages.k_6.dt_0_01.consensus.txt (3.4 MB)

读取 cNMF spectra 与 usage...
  spectra: (6, 1231)  (programs x PACs)
  usage:   (42253, 6)  (cells x programs)
  programs (已统一命名): ['P01', 'P02', 'P03', 'P04', 'P05', 'P06']

计算 spectra 逐 program 的 Z-score...
  spectra_z: (6, 1231)

读取 metadata (h5ad .obs)...
  meta: (42253, 8)  (cells x columns)
  sample     -> 列 'sample'，取值样例: ['cov_01', 'cov_04', 'cov_10', 'cov_11', 'Healthy_07']
  celltype   -> 列 'cell_type'，取值样例

In [19]:
# Cell 4: Layer1
program_direction = compute_program_direction(data, CONFIG)
leading_genes = compute_leading_genes(data, CONFIG, program_direction)
program_direction[["n_fg_genes", "n_bg_genes", "bg_median", "delta_median", "pval", "padj", "direction"]]


计算基因级 wPSI（8793 个基因 x 42253 个细胞）...
  基因检出率过滤(min_detection_rate=0.1): 排除 2788/8793 个基因(wPSI在多数细胞中卡在0/1边界，中位数不可靠)
  过滤后有效基因数: 6005
  cNMF分析涉及的基因全集（pac_universe，基于1231个PAC）大小: 600
  background_scope = 'pac_universe'
  [P01] 前景基因=59, 背景基因=541, delta_median=-0.1267, pval=1.75e-01 -> Proximal
  [P02] 前景基因=62, 背景基因=538, delta_median=+0.0237, pval=6.93e-01 -> Distal
  [P03] 前景基因=46, 背景基因=554, delta_median=+0.0094, pval=8.48e-01 -> Distal
  [P04] 前景基因=58, 背景基因=542, delta_median=-0.1046, pval=1.87e-02 -> Proximal
  [P05] 前景基因=31, 背景基因=569, delta_median=+0.0652, pval=3.38e-01 -> Distal
  [P06] 前景基因=37, 背景基因=563, delta_median=+0.2176, pval=1.31e-01 -> Distal

共 6/6 个 program 完成方向检验
  基因检出率过滤(min_detection_rate=0.1): 排除 2788/8793 个基因(wPSI在多数细胞中卡在0/1边界，中位数不可靠)
  过滤后有效基因数: 6005
  [P01] direction=Proximal, leading genes: ['RAN', 'RPL34', 'OPTN', 'PIK3IP1', 'CD247']...
  [P02] direction=Distal, leading genes: ['ENSG00000268173.3', 'APOBEC3A', 'TRIB1', 'DOK3', 'LRRC25']...
  [P03] direction=Distal, le

,n_fg_genes,n_bg_genes,bg_median,delta_median,pval,padj,direction
program,,,,,,,
P01,59,541,0.255516,-0.126662,0.175065,0.350130,Proximal
P02,62,538,0.232913,0.023693,0.693164,0.831797,Distal
P03,46,554,0.237397,0.009394,0.848233,0.848233,Distal
P04,58,542,0.256468,-0.104642,0.018730,0.112379,Proximal
P05,31,569,0.234955,0.065154,0.338376,0.507564,Distal
P06,37,563,0.225885,0.217642,0.130640,0.350130,Distal


In [20]:
gene_tables, consistency_summary = compute_foreground_gene_directions(data, CONFIG, program_direction)
consistency_summary

# 挑一致性最强的program画出全部前景基因的方向分布（复用已有的plot_leading_genes_bar，
# 之前只用leading_genes传了top_n个基因，这里传全部前景基因，直接看整体分布）
for p in ["P06", "P04"]:
    plot_leading_genes_bar(gene_tables, p, CONFIG["output_root"] + "layer1",
                            direction_label=program_direction.loc[p, "direction"])

  基因检出率过滤(min_detection_rate=0.1): 排除 2788/8793 个基因(wPSI在多数细胞中卡在0/1边界，中位数不可靠)
  过滤后有效基因数: 6005
  [P01] 前景基因59个(非零59个): 与program方向(Proximal)一致 36/59 (61.0%), 符号检验 p=0.059
  [P02] 前景基因62个(非零62个): 与program方向(Distal)一致 34/62 (54.8%), 符号检验 p=0.263
  [P03] 前景基因46个(非零46个): 与program方向(Distal)一致 24/46 (52.2%), 符号检验 p=0.441
  [P04] 前景基因58个(非零58个): 与program方向(Proximal)一致 36/58 (62.1%), 符号检验 p=0.043
  [P05] 前景基因31个(非零31个): 与program方向(Distal)一致 17/31 (54.8%), 符号检验 p=0.360
  [P06] 前景基因37个(非零37个): 与program方向(Distal)一致 24/37 (64.9%), 符号检验 p=0.049
  已保存: /Users/hedanni/Desktop/DEPA_downstream/outputs/covid_19/layer1/layer1_leading_genes_P06.png
  已保存: /Users/hedanni/Desktop/DEPA_downstream/outputs/covid_19/layer1/layer1_leading_genes_P04.png


In [21]:
data_plasma = subset_data_to_celltype(data, CONFIG, "Plasma", min_cells=50)

if data_plasma is not None:
    # 1. pooled方向检验，限定Plasma内部
    program_direction_plasma = compute_program_direction(data_plasma, CONFIG)
    print(program_direction_plasma.loc[["P06"]][["n_fg_genes", "n_bg_genes", "delta_median", "pval", "padj", "direction"]])

    # 2. 符号一致性检验，限定Plasma内部
    leading_genes_plasma = compute_leading_genes(data_plasma, CONFIG, program_direction_plasma)
    gene_tables_plasma, consistency_summary_plasma = compute_foreground_gene_directions(data_plasma, CONFIG, program_direction_plasma)
    print(consistency_summary_plasma.loc[["P06"]])

    # 3. usage分箱趋势，限定Plasma内部（跟原来全细胞版本的图分开存，方便对比）
    summary_p, top_genes_p, r_p, p_p = compute_usage_binned_wpsi(data_plasma, CONFIG, leading_genes_plasma, program="P06")
    plot_usage_binned_trend(
        summary_p, top_genes_p, "P06", r_p, p_p, program_direction_plasma,
        CONFIG["output_root"] + "layer1",
        filename="layer1_usage_binned_trend_P06_within_Plasma.png",
    )

  [subset] 限定 celltype='Plasma'，共 1322 个细胞
计算基因级 wPSI（8793 个基因 x 1322 个细胞）...
  基因检出率过滤(min_detection_rate=0.1): 排除 3133/8793 个基因(wPSI在多数细胞中卡在0/1边界，中位数不可靠)
  过滤后有效基因数: 5660
  cNMF分析涉及的基因全集（pac_universe，基于1231个PAC）大小: 589
  background_scope = 'pac_universe'
  [P01] 前景基因=58, 背景基因=531, delta_median=-0.1069, pval=9.86e-02 -> Proximal
  [P02] 前景基因=59, 背景基因=530, delta_median=+0.0127, pval=4.79e-01 -> Distal
  [P03] 前景基因=44, 背景基因=545, delta_median=+0.0177, pval=9.38e-01 -> Distal
  [P04] 前景基因=56, 背景基因=533, delta_median=-0.0738, pval=5.54e-02 -> Proximal
  [P05] 前景基因=30, 背景基因=559, delta_median=+0.2674, pval=4.30e-02 -> Distal
  [P06] 前景基因=40, 背景基因=549, delta_median=+0.0926, pval=5.41e-01 -> Distal

共 6/6 个 program 完成方向检验
         n_fg_genes  n_bg_genes  delta_median      pval      padj direction
program                                                                    
P06              40         549      0.092615  0.541116  0.649339    Distal
  基因检出率过滤(min_detection_rate=0.1): 排除 3133/8793 个

In [22]:
# Cell 5: Layer1画图
fig_path = plot_program_direction_summary(program_direction, CONFIG, CONFIG["output_root"] + "layer1")

summary, top_genes, r, p = compute_usage_binned_wpsi(data, CONFIG, leading_genes, program="P03")
plot_usage_binned_trend(summary, top_genes, "P03", r, p, program_direction, CONFIG["output_root"] + "layer1")

  已保存: /Users/hedanni/Desktop/DEPA_downstream/outputs/covid_19/layer1/layer1_program_direction_summary.png
  [P03] 15/15 个leading genes用于usage分箱趋势
  [P03] usage vs 每细胞平均wPSI 的Spearman相关: r=-0.073, p=7.51e-51
  已保存: /Users/hedanni/Desktop/DEPA_downstream/outputs/covid_19/layer1/layer1_usage_binned_trend_P03.png


'/Users/hedanni/Desktop/DEPA_downstream/outputs/covid_19/layer1/layer1_usage_binned_trend_P03.png'

In [23]:
summary, top_genes, r, p = compute_usage_binned_wpsi(data, CONFIG, leading_genes, program="P06")
plot_usage_binned_trend(summary, top_genes, "P06", r, p, program_direction, CONFIG["output_root"] + "layer1")

  [P06] 15/15 个leading genes用于usage分箱趋势
  [P06] usage vs 每细胞平均wPSI 的Spearman相关: r=-0.260, p=0.00e+00
  已保存: /Users/hedanni/Desktop/DEPA_downstream/outputs/covid_19/layer1/layer1_usage_binned_trend_P06.png


'/Users/hedanni/Desktop/DEPA_downstream/outputs/covid_19/layer1/layer1_usage_binned_trend_P06.png'

In [24]:
program_direction.loc[["P06", "P04", "P01"]][["n_fg_genes", "n_bg_genes", "delta_median", "pval", "padj"]]

,n_fg_genes,n_bg_genes,delta_median,pval,padj
program,,,,,
P06,37,563,0.217642,0.130640,0.350130
P04,58,542,-0.104642,0.018730,0.112379
P01,59,541,-0.126662,0.175065,0.350130


In [25]:
# Cell 6: Layer2
assoc_results = compute_contrast_association(data, CONFIG, program_direction=program_direction)
assoc_results["disease"]

  contrast 'celltype_identity' (auto_one_vs_rest_all, column='cell_type') 自动展开为 14 条 [category=identity]: ['Erythroid', 'HSPC', 'Memory_B', 'NK', 'Naive_B', 'Naive_CD4T', 'Plasma', 'Platelet', 'cDC1', 'cDC2', 'cMono', 'iMono', 'ncMono', 'pDC']

=== contrast 'disease' [disease] (column='group', type='pairwise') ===
  Healthy: 28107 细胞  |  Severe_COVID: 14146 细胞
  最强关联: P03 (|AUC-0.5|=0.240)

=== contrast 'Erythroid' [identity] (column='cell_type', type='one_vs_rest') ===
  not_Erythroid: 41859 细胞  |  Erythroid: 394 细胞
  最强关联: P01 (|AUC-0.5|=0.389)

=== contrast 'HSPC' [identity] (column='cell_type', type='one_vs_rest') ===
  not_HSPC: 41659 细胞  |  HSPC: 594 细胞
  最强关联: P05 (|AUC-0.5|=0.269)

=== contrast 'Memory_B' [identity] (column='cell_type', type='one_vs_rest') ===
  not_Memory_B: 42071 细胞  |  Memory_B: 182 细胞
  最强关联: P03 (|AUC-0.5|=0.271)

=== contrast 'NK' [identity] (column='cell_type', type='one_vs_rest') ===
  not_NK: 32831 细胞  |  NK: 9422 细胞
  最强关联: P01 (|AUC-0.5|=0.384)

=== 

,auc,lr_coef,dt_importance,xgb_importance,n_group_a,n_group_b,direction
program,,,,,,,
P03,0.739589,0.990865,0.524290,0.319654,28107,14146,Distal
P02,0.318108,-1.287987,0.385757,0.312507,28107,14146,Distal
P05,0.414561,-0.472421,0.000000,0.116007,28107,14146,Distal
P04,0.427973,-0.423852,0.055655,0.120773,28107,14146,Proximal
P01,0.538093,0.046750,0.034299,0.079854,28107,14146,Proximal
P06,0.499740,0.080039,0.000000,0.051205,28107,14146,Distal


In [26]:
# Cell 7: Layer2画图
CONFIG = yaml.safe_load(open(PROJECT_ROOT / "configs" / "covid19.yaml"))
assoc_results = compute_contrast_association(data, CONFIG, program_direction=program_direction)
plot_association_heatmap(assoc_results, CONFIG, CONFIG["output_root"] + "layer2")
plot_program_umap_grid(data, CONFIG, CONFIG["output_root"] + "layer2")

  contrast 'celltype_identity' (auto_one_vs_rest_all, column='cell_type') 自动展开为 14 条 [category=identity]: ['Erythroid', 'HSPC', 'Memory_B', 'NK', 'Naive_B', 'Naive_CD4T', 'Plasma', 'Platelet', 'cDC1', 'cDC2', 'cMono', 'iMono', 'ncMono', 'pDC']

=== contrast 'disease' [disease] (column='group', type='pairwise') ===
  Healthy: 28107 细胞  |  Severe_COVID: 14146 细胞
  最强关联: P03 (|AUC-0.5|=0.240)

=== contrast 'Erythroid' [identity] (column='cell_type', type='one_vs_rest') ===
  not_Erythroid: 41859 细胞  |  Erythroid: 394 细胞
  最强关联: P01 (|AUC-0.5|=0.389)

=== contrast 'HSPC' [identity] (column='cell_type', type='one_vs_rest') ===
  not_HSPC: 41659 细胞  |  HSPC: 594 细胞
  最强关联: P05 (|AUC-0.5|=0.269)

=== contrast 'Memory_B' [identity] (column='cell_type', type='one_vs_rest') ===
  not_Memory_B: 42071 细胞  |  Memory_B: 182 细胞
  最强关联: P03 (|AUC-0.5|=0.271)

=== contrast 'NK' [identity] (column='cell_type', type='one_vs_rest') ===
  not_NK: 32831 细胞  |  NK: 9422 细胞
  最强关联: P01 (|AUC-0.5|=0.384)

=== 

'outputs/covid_19/layer2/layer2_program_umap_grid.png'

In [27]:
# Layer2 cell（接在 compute_contrast_association 之后）
result = compute_gene_celltype_wpsi(data, CONFIG, leading_genes)
if result is not None:
    wpsi_matrix, gene_program_map, gene_symbol_map = result
    plot_gene_celltype_heatmap(wpsi_matrix, gene_program_map, gene_symbol_map, CONFIG["output_root"] + "layer2")

/Users/hedanni/Desktop/DEPA_downstream/depa_downstream/layer2_association.py:485: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_mean = row.groupby(meta.loc[common_cells, groupby_col]).mean()
/Users/hedanni/Desktop/DEPA_downstream/depa_downstream/layer2_association.py:485: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_mean = row.groupby(meta.loc[common_cells, groupby_col]).mean()
/Users/hedanni/Desktop/DEPA_downstream/depa_downstream/layer2_association.py:485: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=Fals

  gene x cell_type wPSI矩阵: 86 个基因 x 14 个分组
  已保存: outputs/covid_19/layer2/layer2_gene_celltype_heatmap.png


In [28]:

#import mygene
#import pandas as pd

#ensg_ids = data["feature_anno"]["gene_id"].str.split(".").str[0].unique().tolist()
#mg = mygene.MyGeneInfo()
#result = mg.querymany(ensg_ids, scopes="ensembl.gene", fields="symbol", species="human")

#mapping_df = pd.DataFrame(result).dropna(subset=["symbol"])[["query", "symbol"]]
#mapping_df.columns = ["gene_id", "symbol"]
#mapping_df.to_csv(PROJECT_ROOT / "configs" / "covid19_gene_symbol_map.csv", index=False)
#print(f"映射成功 {len(mapping_df)} / {len(ensg_ids)} 个基因")

In [29]:
#Layer3
program_gene_sets = compute_program_gene_sets(data, CONFIG)
markers = select_program_markers(program_gene_sets, data, CONFIG, top_n=20, spec_min_z=0.5)
markers["P03"][["gene_id", "symbol"]].head(10)   # 先确认symbol转换成功了

  [P01] 核心基因集大小: 65
  [P02] 核心基因集大小: 66
  [P03] 核心基因集大小: 50
  [P04] 核心基因集大小: 64
  [P05] 核心基因集大小: 33
  [P06] 核心基因集大小: 41
  [P01] 候选基因65个 -> 专属marker20个 (spec_min_z=0.5)
  [P02] 候选基因66个 -> 专属marker20个 (spec_min_z=0.5)
  [P03] 候选基因50个 -> 专属marker10个 (spec_min_z=0.5)
  [P04] 候选基因64个 -> 专属marker20个 (spec_min_z=0.5)
  [P05] 候选基因33个 -> 专属marker14个 (spec_min_z=0.5)
  [P06] 候选基因41个 -> 专属marker15个 (spec_min_z=0.5)


,gene_id,symbol
0,ENSG00000158481.13,CD1C
1,ENSG00000132514.14,CLEC10A
2,ENSG00000125538.12,IL1B
3,ENSG00000167552.15,TUBA1A
4,ENSG00000143110.12,C1orf162
5,ENSG00000015475.19,BID
6,ENSG00000073756.13,PTGS2
7,ENSG00000165312.7,OTUD1
8,ENSG00000127507.18,ADGRE2
9,ENSG00000185022.12,MAFF


In [30]:
#Layer3
gene_lists = {f"{p}_markers": markers[p]["symbol"].tolist() for p in data["programs"]}
enrichment = run_enrichment(gene_lists, CONFIG)
plot_enrichment_bubble(enrichment, CONFIG["output_root"] + "layer3")

  [P01_markers] 20个基因 -> 355条富集term，最强: TNF-alpha Signaling via NF-kB (padj=7.05e-04)
  [P02_markers] 过滤掉 1 个仍是ENSG格式(未成功映射到symbol)的基因，剩余 19 个送去Enrichr查询
  [P02_markers] 19个基因 -> 528条富集term，最强: Neutrophil Degranulation R-HSA-6798695 (padj=7.25e-07)
  [P03_markers] 10个基因 -> 418条富集term，最强: Regulation Of Prostaglandin Biosynthetic Process (GO:0031392) (padj=4.33e-04)
  [P04_markers] 20个基因 -> 366条富集term，最强: Neutrophil extracellular trap formation (padj=1.03e-03)
  [P05_markers] 14个基因 -> 179条富集term，最强: Sphingolipid metabolism (padj=5.78e-03)
  [P06_markers] 15个基因 -> 190条富集term，最强: IRE1alpha Activates Chaperones R-HSA-381070 (padj=2.31e-04)
  已保存: outputs/covid_19/layer3/layer3_enrichment_bubble.png


'outputs/covid_19/layer3/layer3_enrichment_bubble.png'

In [31]:
gene_lists_all = {}
for p in data["programs"]:
    # 用compute_program_gene_sets里算出的核心基因集（不要求排他性）
    gene_ids = program_gene_sets["gene_sets"][p]
    symbols = [
        data["gene_symbol_map"].loc[gid.split(".")[0]] 
        if gid.split(".")[0] in data["gene_symbol_map"].index 
        else gid 
        for gid in gene_ids
    ]
    gene_lists_all[f"{p}_all"] = symbols

enrichment_all = run_enrichment(gene_lists_all, CONFIG)
plot_enrichment_bubble(
    enrichment_all,
    CONFIG["output_root"] + "layer3",
    top_n=6,
    filename="layer3_enrichment_bubble_all.png",
)

  [P01_all] 过滤掉 2 个仍是ENSG格式(未成功映射到symbol)的基因，剩余 63 个送去Enrichr查询
  [P01_all] 63个基因 -> 1209条富集term，最强: Allograft Rejection (padj=3.47e-07)
  [P02_all] 过滤掉 1 个仍是ENSG格式(未成功映射到symbol)的基因，剩余 65 个送去Enrichr查询
  [P02_all] 65个基因 -> 1227条富集term，最强: Neutrophil Degranulation R-HSA-6798695 (padj=2.32e-26)
  ⚠ [P03_all] Enrichr查询失败: HTTPSConnectionPool(host='maayanlab.cloud', port=443): Read timed out. (read timeout=30)
  ⚠ [P04_all] Enrichr查询失败: HTTPSConnectionPool(host='maayanlab.cloud', port=443): Max retries exceeded with url: /Enrichr/addList (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1017)')))
  [P05_all] 过滤掉 1 个仍是ENSG格式(未成功映射到symbol)的基因，剩余 32 个送去Enrichr查询
  [P05_all] 32个基因 -> 628条富集term，最强: Apical Surface (padj=3.95e-02)
  [P06_all] 41个基因 -> 659条富集term，最强: Protein processing in endoplasmic reticulum (padj=1.57e-12)
  已保存: outputs/covid_19/layer3/layer3_enrichment_bubble_all.png


'outputs/covid_19/layer3/layer3_enrichment_bubble_all.png'

In [32]:
driver_genes = extract_enrichment_driver_genes(enrichment, leading_genes, data, CONFIG, top_n_terms=5)
driver_genes[driver_genes["in_leading_genes"]].head(10)

for p in data["programs"]:
    plot_driver_gene_heatmap(driver_genes, wpsi_matrix, gene_symbol_map, CONFIG["output_root"] + "layer3", program=p)

  [P01] top5条term共16个驱动基因(去重前)，其中12个同时是leading gene
  [P02] top5条term共34个驱动基因(去重前)，其中3个同时是leading gene
  [P03] top5条term共13个驱动基因(去重前)，其中5个同时是leading gene
  [P04] top5条term共17个驱动基因(去重前)，其中6个同时是leading gene
  [P05] top5条term共8个驱动基因(去重前)，其中5个同时是leading gene
  [P06] top5条term共14个驱动基因(去重前)，其中9个同时是leading gene
  已保存: outputs/covid_19/layer3/layer3_driver_gene_heatmap_P01.png
  已保存: outputs/covid_19/layer3/layer3_driver_gene_heatmap_P02.png
  已保存: outputs/covid_19/layer3/layer3_driver_gene_heatmap_P03.png
  已保存: outputs/covid_19/layer3/layer3_driver_gene_heatmap_P04.png
  已保存: outputs/covid_19/layer3/layer3_driver_gene_heatmap_P05.png
  已保存: outputs/covid_19/layer3/layer3_driver_gene_heatmap_P06.png


In [33]:
retry = run_enrichment({"P06_markers": gene_lists["P06_markers"]}, CONFIG)
enrichment["P06_markers"] = retry["P06_markers"]

  [P06_markers] 15个基因 -> 190条富集term，最强: IRE1alpha Activates Chaperones R-HSA-381070 (padj=2.31e-04)
